<a href="https://colab.research.google.com/github/yaaaaaolaaaaa/polinagogokina/blob/main/%D0%93%D0%BE%D0%B3%D0%BE%D0%BA%D0%B8%D0%BD%D0%B0_fine_tuning_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [1]:
!pip install transformers datasets evaluate accelerate -q

import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
import evaluate
from transformers import pipeline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dataset = load_dataset("fancyzhx/ag_news")
print(f"Train samples: {len(dataset['train'])}, Test samples: {len(dataset['test'])}")

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
).to(device)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize, batched=True)
train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

training_args = TrainingArguments(
    output_dir="./ag_news_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=500,
    report_to="none",
    push_to_hub=False,
)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

test_results = trainer.evaluate()
print(f"\nTest accuracy: {test_results['eval_accuracy']:.4f}")

model_save_path = "./ag_news_model"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model saved to {model_save_path}")

classifier = pipeline(
    "text-classification",
    model=model_save_path,
    tokenizer=model_save_path,
    device=0 if torch.cuda.is_available() else -1
)

id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

new_articles = [
    "The national football team won the championship match in a dramatic penalty shootout.",
    "The Federal Reserve announced a surprise interest rate cut to boost the slowing economy.",
    "Scientists have discovered a new exoplanet that may contain liquid water, a key ingredient for life."
]

print("\n" + "="*50)
print("Testing on new articles:")
print("="*50)
for article in new_articles:
    result = classifier(article)[0]
    label_id = int(result['label'].split('_')[-1])
    label_name = id2label[label_id]
    confidence = result['score']
    print(f"\nArticle: {article}")
    print(f"Predicted class: {label_name} (confidence: {confidence:.4f})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

...


Test accuracy: 0.9474


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./ag_news_model


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Testing on new articles:

Article: The national football team won the championship match in a dramatic penalty shootout.
Predicted class: Sports (confidence: 0.6664)

Article: The Federal Reserve announced a surprise interest rate cut to boost the slowing economy.
Predicted class: Business (confidence: 0.9917)

Article: Scientists have discovered a new exoplanet that may contain liquid water, a key ingredient for life.
Predicted class: Sci/Tech (confidence: 0.9721)
